# Early Sepsis Alert System (6-Hour Ahead Prediction)

Hackathon Prototype

This project develops a clinically realistic early warning system
for sepsis using ICU time-series data.

Focus: Reduce alert fatigue while maintaining early detection capability.

## Load Data

In [103]:
import pandas as pd

baseline_df = pd.read_csv(
    "/content/drive/MyDrive/physionet/baseline_dataset.psv",
    sep="|"
)

print("Shape:", baseline_df.shape)
print("Patients:", baseline_df["PatientID"].nunique())
print("Positive rate (%):", baseline_df["FutureSepsis"].mean() * 100)

Shape: (790215, 14)
Patients: 20336
Positive rate (%): 1.1255164733648437


## Patient-Wise Train/Test Split

In [104]:
from sklearn.model_selection import train_test_split

patients = baseline_df['PatientID'].unique()

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42
)

train_df = baseline_df[baseline_df['PatientID'].isin(train_patients)].copy()
test_df  = baseline_df[baseline_df['PatientID'].isin(test_patients)].copy()

print("Train patients:", train_df['PatientID'].nunique())
print("Test patients:", test_df['PatientID'].nunique())

Train patients: 16268
Test patients: 4068


## Missing Handling

In [105]:
lab_cols = ['Lactate','WBC','Creatinine','Platelets']
vitals = ['HR','O2Sat','SBP','MAP','Resp','Temp']

# Missing indicators
for col in lab_cols:
    train_df[col + '_missing'] = train_df[col].isnull().astype(int)
    test_df[col + '_missing']  = test_df[col].isnull().astype(int)

# Forward fill vitals per patient
train_df[vitals] = train_df.groupby('PatientID')[vitals].ffill()
test_df[vitals]  = test_df.groupby('PatientID')[vitals].ffill()

# Median fill (train statistics only)
vital_medians = train_df[vitals].median()
train_df[vitals] = train_df[vitals].fillna(vital_medians)
test_df[vitals]  = test_df[vitals].fillna(vital_medians)

## **Feature Engineering** (Final Version - v6)

In [106]:
import numpy as np
# Delta6
for col in vitals:
    train_df[col + '_delta6'] = (
        train_df.groupby('PatientID')[col].shift(0) -
        train_df.groupby('PatientID')[col].shift(6)
    ).fillna(0)

    test_df[col + '_delta6'] = (
        test_df.groupby('PatientID')[col].shift(0) -
        test_df.groupby('PatientID')[col].shift(6)
    ).fillna(0)

# Delta1
for col in vitals:
    train_df[col + '_delta1'] = (
        train_df.groupby('PatientID')[col].shift(0) -
        train_df.groupby('PatientID')[col].shift(1)
    ).fillna(0)

    test_df[col + '_delta1'] = (
        test_df.groupby('PatientID')[col].shift(0) -
        test_df.groupby('PatientID')[col].shift(1)
    ).fillna(0)

# Volatility (6-hour std)
for col in vitals:
    train_df[col + '_roll6_std'] = (
        train_df.groupby('PatientID')[col]
        .rolling(window=6, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
    ).fillna(0)

    test_df[col + '_roll6_std'] = (
        test_df.groupby('PatientID')[col]
        .rolling(window=6, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
    ).fillna(0)

train_df = train_df.copy()
test_df  = test_df.copy()

# Patient Baseline Deviation
baseline_window = 6

for col in vitals:

    baseline_train = (
        train_df.groupby("PatientID")[col]
        .transform(lambda x: x.iloc[:baseline_window].mean())
    )

    baseline_test = (
        test_df.groupby("PatientID")[col]
        .transform(lambda x: x.iloc[:baseline_window].mean())
    )

    train_df[col + "_baseline_dev"] = train_df[col] - baseline_train
    test_df[col + "_baseline_dev"] = test_df[col] - baseline_test

# Lab presence in last 6 hours

for col in lab_cols:

    train_df[col + "_recent_test"] = (
        train_df.groupby("PatientID")[col]
        .apply(lambda x: x.notnull().rolling(6, min_periods=1).max())
        .reset_index(level=0, drop=True)
    )

    test_df[col + "_recent_test"] = (
        test_df.groupby("PatientID")[col]
        .apply(lambda x: x.notnull().rolling(6, min_periods=1).max())
        .reset_index(level=0, drop=True)
    )


# Clinical Ratio Features


train_df["shock_index"] = train_df["HR"] / (train_df["SBP"] + 1)
test_df["shock_index"]  = test_df["HR"] / (test_df["SBP"] + 1)

train_df["resp_o2_ratio"] = train_df["Resp"] / (train_df["O2Sat"] + 1)
test_df["resp_o2_ratio"]  = test_df["Resp"] / (test_df["O2Sat"] + 1)

train_df["map_hr_ratio"] = train_df["MAP"] / (train_df["HR"] + 1)
test_df["map_hr_ratio"]  = test_df["MAP"] / (test_df["HR"] + 1)


# Clinical Threshold Flags


train_df["tachycardia"] = (train_df["HR"] > 100).astype(int)
test_df["tachycardia"]  = (test_df["HR"] > 100).astype(int)

train_df["hypotension"] = (train_df["SBP"] < 90).astype(int)
test_df["hypotension"]  = (test_df["SBP"] < 90).astype(int)

train_df["tachypnea"] = (train_df["Resp"] > 22).astype(int)
test_df["tachypnea"]  = (test_df["Resp"] > 22).astype(int)


## **Train Final Model** (HistGradientBoosting)

In [107]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

X_train = train_df.drop(columns=['FutureSepsis','PatientID'])
y_train = train_df['FutureSepsis']

X_test = test_df.drop(columns=['FutureSepsis','PatientID'])
y_test = test_df['FutureSepsis']

hgb_model = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.1,
    max_iter=200,
    random_state=42
)

hgb_model.fit(X_train, y_train)

y_pred_proba = hgb_model.predict_proba(X_test)[:,1]

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC:", average_precision_score(y_test, y_pred_proba))

ROC-AUC: 0.756004458507216
PR-AUC: 0.0399172981371418


## **Alert System Design** (Final Deployment Logic)

In [108]:
threshold = 0.045
cooldown_hours = 3

# uncertainty band
uncertainty_low = 0.035
uncertainty_high = 0.055

test_df_alert = test_df.copy()
test_df_alert["prob"] = y_pred_proba

test_df_alert = test_df_alert.sort_values(["PatientID", "ICULOS"])


# Uncertainty rejection step

test_df_alert["uncertain"] = (
    (test_df_alert["prob"] > uncertainty_low) &
    (test_df_alert["prob"] < uncertainty_high)
)

# remove uncertain predictions
test_df_alert["prob_filtered"] = test_df_alert["prob"]
test_df_alert.loc[test_df_alert["uncertain"], "prob_filtered"] = 0



# High risk flag

test_df_alert["high_risk"] = (
    test_df_alert["prob_filtered"] >= threshold
).astype(int)


# 2-hour persistence rule

test_df_alert["high_risk_prev"] = (
    test_df_alert.groupby("PatientID")["high_risk"]
    .shift(1)
    .fillna(0)
)

test_df_alert["alert_raw"] = (
    (test_df_alert["high_risk"] == 1) &
    (test_df_alert["high_risk_prev"] == 1)
).astype(int)



# Cooldown Logic

test_df_alert["alert"] = 0

for pid in test_df_alert["PatientID"].unique():

    patient_rows = test_df_alert[test_df_alert["PatientID"] == pid]

    last_alert_time = -999

    for idx, row in patient_rows.iterrows():

        if row["alert_raw"] == 1:

            if row["ICULOS"] - last_alert_time >= cooldown_hours:

                test_df_alert.loc[idx, "alert"] = 1
                last_alert_time = row["ICULOS"]



# Patient-level evaluation

patient_alerts = test_df_alert.groupby("PatientID").agg({
    "alert": "max",
    "FutureSepsis": "max"
}).reset_index()

TP = ((patient_alerts["alert"] == 1) &
      (patient_alerts["FutureSepsis"] == 1)).sum()

FP = ((patient_alerts["alert"] == 1) &
      (patient_alerts["FutureSepsis"] == 0)).sum()

FN = ((patient_alerts["alert"] == 0) &
      (patient_alerts["FutureSepsis"] == 1)).sum()

precision = TP / (TP + FP)
recall = TP / (TP + FN)

print("Final Patient-Level Precision:", precision)
print("Final Patient-Level Recall:", recall)

Final Patient-Level Precision: 0.625
Final Patient-Level Recall: 0.42071197411003236


# **Conclusion (Markdown)**

## Final System Summary

We developed a clinically inspired early warning system for **6-hour ahead sepsis prediction** using ICU time-series data.

The model combines temporal feature engineering with a tree-based machine learning model to capture physiological deterioration patterns before clinical onset.

### Key Design Components

• **HistGradientBoosting classifier** trained on engineered temporal features  
• **Patient-wise train/test split** to avoid information leakage  
• **Temporal dynamics modeling**, including:
  - short-term changes (Δ1 hour)
  - medium-term trends (Δ6 hours)
  - physiological instability (rolling standard deviation)

• **Patient baseline normalization**, allowing the model to detect deviations from each patient's normal state

• **Clinical behaviour signals**, such as recent laboratory test activity

• **Clinically meaningful features**, including the **Shock Index (HR / SBP)** to capture circulatory stress

• **Alert system design** incorporating:
  - probability thresholding
  - 2-hour persistence rule
  - alert cooldown mechanism

These mechanisms reduce noisy spikes and help generate **clinically realistic alerts**.

---

## Final Deployment Performance (Patient-Level)

Precision ≈ **62%**  
Recall ≈ **42%**

Interpretation:

• When the system triggers an alert, **about 6 out of 10 alerts correspond to real sepsis events**  
• The system successfully identifies **~42% of patients who will develop sepsis within the next 6 hours**

The alert filtering strategy significantly **reduces false alarms while preserving early detection capability**, addressing the common ICU challenge of alert fatigue.

---

## Key Takeaway

Rather than maximizing raw model metrics alone, this project focuses on building a **practical early-warning pipeline**:


This approach produces a **more reliable and interpretable early sepsis detection system**, making it better suited for real ICU deployment scenarios.